# 02 — Baseline Models: Logistic Regression vs Random Forest

This notebook trains two baseline classifiers, evaluates them on the validation set,
and produces an ablation table and Precision-Recall comparison.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    precision_recall_curve,
)

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocess import preprocess

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
print('All imports OK')

In [ ]:
# ── Preprocess & get splits ──────────────────────────────────────
csv_path = os.path.join(PROJECT_ROOT, 'data', 'creditcard_2023.csv')
X_train, X_val, X_test, y_train, y_val, y_test = preprocess(csv_path)

print(f'X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')
print(f'Fraud ratio — train: {y_train.mean():.4f}  val: {y_val.mean():.4f}')

In [ ]:
# ── Train Logistic Regression ────────────────────────────────────
lr = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)
lr.fit(X_train, y_train)
print('Logistic Regression trained.')

In [ ]:
# ── Train Random Forest ──────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print('Random Forest trained.')

In [ ]:
# ── Logistic Regression: Classification Report & Confusion Matrix ─
y_pred_lr = lr.predict(X_val)
y_proba_lr = lr.predict_proba(X_val)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_val, y_pred_lr, target_names=['Legit', 'Fraud']))
print('Confusion Matrix:')
cm_lr = confusion_matrix(y_val, y_pred_lr)
print(cm_lr)

In [ ]:
# ── LR Confusion Matrix Heatmap ─────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# ── Random Forest: Classification Report & Confusion Matrix ──────
y_pred_rf = rf.predict(X_val)
y_proba_rf = rf.predict_proba(X_val)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_val, y_pred_rf, target_names=['Legit', 'Fraud']))
print('Confusion Matrix:')
cm_rf = confusion_matrix(y_val, y_pred_rf)
print(cm_rf)

In [ ]:
# ── RF Confusion Matrix Heatmap ──────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Random Forest — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# ── Precision-Recall Curves: LR vs RF ────────────────────────────
prec_lr, rec_lr, _ = precision_recall_curve(y_val, y_proba_lr)
prec_rf, rec_rf, _ = precision_recall_curve(y_val, y_proba_rf)

ap_lr = average_precision_score(y_val, y_proba_lr)
ap_rf = average_precision_score(y_val, y_proba_rf)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(rec_lr, prec_lr, label=f'Logistic Regression (AUC-PR={ap_lr:.4f})', linewidth=2)
ax.plot(rec_rf, prec_rf, label=f'Random Forest (AUC-PR={ap_rf:.4f})', linewidth=2)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve: LR vs Random Forest')
ax.legend(loc='lower left', fontsize=11)
ax.set_xlim([0, 1.02])
ax.set_ylim([0, 1.05])
plt.tight_layout()

# Save figure
fig_path = os.path.join(PROJECT_ROOT, 'reports', 'figures', 'pr_curve_baseline.png')
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
fig.savefig(fig_path, dpi=150)
print(f'Saved to {fig_path}')
plt.show()

In [ ]:
# ── Ablation Table: All 7 Metrics ────────────────────────────────
def compute_metrics(y_true, y_pred, y_proba):
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='binary'),
        'recall':    recall_score(y_true, y_pred, average='binary'),
        'f1':        f1_score(y_true, y_pred, average='binary'),
        'roc_auc':   roc_auc_score(y_true, y_proba),
        'auc_pr':    average_precision_score(y_true, y_proba),
        'mcc':       matthews_corrcoef(y_true, y_pred),
    }

lr_metrics = compute_metrics(y_val, y_pred_lr, y_proba_lr)
rf_metrics = compute_metrics(y_val, y_pred_rf, y_proba_rf)

ablation_df = pd.DataFrame(
    [lr_metrics, rf_metrics],
    index=['Logistic Regression', 'Random Forest']
)
ablation_df.index.name = 'model'

# Display with 4 decimal places
ablation_df.style.format('{:.4f}')

## Initial Metric Report

Random Forest outperforms Logistic Regression across every metric, most notably on AUC-PR (0.9995 vs 0.9946) and precision (0.9987 vs 0.9777), indicating far fewer false alarms while maintaining strong fraud detection.
The confusion matrices reveal that Logistic Regression misses 2,048 fraudulent transactions (false negatives) compared to Random Forest's 1,141, meaning LR lets nearly twice as many frauds slip through undetected.
AUC-PR is chosen as the primary metric over accuracy because, in real-world fraud scenarios with imbalanced classes, accuracy can be misleadingly high simply by predicting the majority class; AUC-PR instead directly measures the trade-off between catching frauds (recall) and avoiding false alarms (precision) across all decision thresholds.
XGBoost is expected to improve upon Random Forest by leveraging sequential boosting to correct the remaining false negatives and achieve an even tighter precision-recall trade-off, particularly at high-recall operating points.